# Importando Bibliotecas

In [77]:
import pandas as pd
import numpy as np
import requests
from io import StringIO

# Importando a base de dados do GitHub

In [ ]:
def carregar_csv_github(url_raw: str) -> pd.DataFrame:
    """
    Baixa um CSV público do GitHub e retorna como DataFrame pandas.
    url_raw: Caminho no GitHub para o arquivo .CSV
    """
    response = requests.get(url_raw, timeout=30)
    response.raise_for_status()
    
    # 1. Lê o arquivo pulando os metadados do topo
    df_lido = pd.read_csv(
        StringIO(response.text), 
        sep=';', 
        skiprows=1, 
        on_bad_lines='skip'
    )
    
    # 2. Transforma a string textual 'null' no valor NaN real do sistema
    df_limpo = df_lido.replace('null', np.nan)
    
    # 3. Retorna o DataFrame perfeitamente estruturado
    return df_limpo

# Execução do código
csv_path = "https://raw.githubusercontent.com/Jobanu/Projeto-Final-DS-PY-004/main/Dados/V_OCORRENCIA_AMPLA.csv"
df = carregar_csv_github(csv_path)

print(f"Carregadas {len(df)} ocorrências")
df.head()


# Tratamento das colunas dos tipos de Lesões - (NaN para 0)

A razão para escolhermos apenas as colunas de lesões e não o DataFrame inteiro é que o significado do "vazio" (NaN) muda completamente dependendo do tipo de dado da coluna. Substituir tudo por zero de forma automática geraria graves erros de lógica e distorções na análise.

1. Colunas de Texto (Strings) ficariam sem sentido

    - Operador: O responsável pelo voo passaria a se chamar 0 (o Pandas não saberia que ali deveria ser "Desconhecido")
    Matricula: O prefixo do avião viraria 0.

2. Colunas Numéricas Específicas seriam corrompidas, pois onúmero 0 possui um valor matemático real em engenharia aeronáutica.

    - PMD (Peso Máximo de Decolagem): Se uma aeronave experimental não teve o peso registrado, colocar 0 faria o Pandas calcular que o avião pesa zero quilos, o que estragaria qualquer média de peso que você tentasse fazer no projeto.
    - Numero_de_Assentos: Colocar 0 indicaria que o avião voava sem nenhum banco dentro, quando na verdade o dado apenas não foi coletado.

3. Nas colunas de Lesões, o 0 mantém a lógica perfeitaNas colunas de contagem de pessoas (Lesoes_Fatais, Lesoes_Graves), assumir que o vazio equivale a 0 é seguro e correto para o modelo de dados, pois:

    - Se o relatório não mencionou feridos, a hipótese estatística padrão é que zero pessoas se machucaram naquela categoria específica.
    - Permite converter a coluna de Float (número quebrado) para Int (número inteiro), já que não existem "1.5 pessoas mortas".

In [ ]:
# 1. Lista com as colunas de lesões e contagem da base
colunas_lesoes = [
    'Lesoes_Fatais_Tripulantes', 'Lesoes_Fatais_Passageiros', 'Lesoes_Fatais_Terceiros',
    'Lesoes_Graves_Tripulantes', 'Lesoes_Graves_Passageiros', 'Lesoes_Graves_Terceiros',
    'Lesoes_Leves_Tripulantes', 'Lesoes_Leves_Passageiros', 'Lesoes_Leves_Terceiros',
    'Ilesos_Tripulantes', 'Ilesos_Passageiros',
    'Lesoes_Desconhecidas_Tripulantes', 'Lesoes_Desconhecidas_Passageiros', 'Lesoes_Desconhecidas_Terceiros'
]

# --- ANTES ---
print("=== 1. QUANTIDADE DE NaNs ANTES DA LIMPEZA ===")
# Soma quantos NaNs existem apenas nas colunas da lista
print(df[colunas_lesoes].isna().sum())
print("-" * 50)


# 2. Criar a função de preenchimento
def preencher_nan_lesoes(dataframe: pd.DataFrame, lista_colunas: list) -> pd.DataFrame:
    """
    Substitui os valores NaN das colunas de lesões pelo número zero (0).
    Aplica a conversão para tipo inteiro (Int64), ideal para contagens.
    """
    # Faz uma cópia para não alterar o DataFrame original por acidente
    df_copia = dataframe.copy()
    
    # Preenche os NaNs com 0 e converte para tipo numérico inteiro
    df_copia[lista_colunas] = df_copia[lista_colunas].fillna(0).astype('int64')
    
    return df_copia


# 3. Aplicar a função no seu DataFrame
df_tratado = preencher_nan_lesoes(df, colunas_lesoes)


# --- DEPOIS ---
print("=== 2. QUANTIDADE DE NaNs DEPOIS DA LIMPEZA ===")
print(df_tratado[colunas_lesoes].isna().sum())
print("-" * 50)

